# Python: Sample Selection Models with a Control Function Correction

In this example we illustrate how a new `DoubleML` model class, `DoubleMLSSCF`, can be used to estimate the **sample selection bias coefficient** $\theta_0$ in a high-dimensional sample selection model, and how the resulting bias correction feeds into variable selection in the outcome equation.

The estimation is based on a simulated data generating process from Chen, Lioui and Scaillet (*Green Silence: Double machine learning carbon emissions under sample selection bias*). The empirical data of that paper (corporate carbon emissions from a commercial vendor) cannot be redistributed, so this notebook works entirely with the simulated DGP shipped with the model class.

## The model

Consider an outcome (variable selection) equation and a selection (participation) equation

$$
\begin{align}
Y_i &= g_0(X_i) + \varepsilon_i, \\
D_i &= \mathbb{1}\{Z_i'\beta_0 + v_i > 0\},
\end{align}
$$

where $Y_i$ is observed **only if** $D_i = 1$, and $Z_i = (X_i, U_i)$ contains the outcome covariates $X_i$ plus additional variables $U_i$ which shift participation but are excluded from the outcome equation. In the motivating application $Y_i$ is (log) carbon emissions and $D_i$ indicates whether firm $i$ voluntarily discloses them.

Under joint normality of $(\varepsilon_i, v_i)$, the selection bias in the observed subsample is $\mathbb{E}[\varepsilon_i \mid X_i, Z_i, D_i = 1] = \theta_0 h_i$, with the inverse Mills ratio $h_i = \varphi(Z_i'\beta_0)/\Phi(Z_i'\beta_0)$, so that the observed outcome equation becomes

$$
Y_i = g_0(X_i) + \theta_0 h_i + u_i, \qquad \mathbb{E}[u_i \mid X_i] = 0 .
$$

The parameter of interest is $\theta_0 = \sigma_{\varepsilon v}$. It is the object that decides whether outcomes observed in a self-selected subsample may be used to impute outcomes for the non-selected units: if $\theta_0 \neq 0$, naive imputation is systematically biased.

## The Neyman-orthogonal score

Estimation is based on the linear score

$$
\psi(W; \theta, \eta) = \big(Y - g(X) - \theta h\big)\big(h - m(X)\big), \qquad \eta = (h, g, m),
$$

evaluated on the selected subpopulation $\{D = 1\}$, i.e. $\psi_a = -h(h - m)$ and $\psi_b = (Y - g)(h - m)$. The nuisance elements are

| nuisance | learner | target |
| --- | --- | --- |
| $\pi(Z) = P(D = 1 \mid Z)$ | `ml_pi` (classifier) | participation propensity score |
| $g(X) = \mathbb{E}[Y \mid X, D = 1]$ | `ml_g` (regressor) | outcome regression on the selected sample |
| $m(X) = \mathbb{E}[h \mid X, D = 1]$ | `ml_m` (regressor) | projection of the inverse Mills ratio |

Two remarks.

**The inverse Mills ratio is link free.** Since $\pi_i = \Phi(Z_i'\beta_0)$, the linear index can be recovered from the propensity score through the probit link, $Z_i'\beta_0 = \Phi^{-1}(\pi_i)$, so that

$$
h_i = \frac{\varphi\big(\Phi^{-1}(\pi_i)\big)}{\pi_i}.
$$

Any probabilistic classifier can therefore be used for the selection equation; a probit specification is just the special case. Note also that $h_i$ diverges as $\pi_i \to 0$, so strict overlap is a genuine identification requirement here, not merely numerical hygiene — the propensity scores are trimmed accordingly.

**Orthogonalization is what identifies $\theta_0$.** The term $h - m(X)$ partials out the variation in the selection process attributable to $X$, so that identification comes from the exclusion restrictions $Z \setminus X$ and from the nonlinearity of the inverse Mills ratio. Correspondingly, $-J_0 = \mathbb{E}[(h - m(X))^2 \mid D = 1]$: precision is driven by the "surprise" variation in the disclosure decision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, LogisticRegression

from doubleml_sscf import DoubleMLSSCF, make_green_silence_data

np.random.seed(3141)

## Data

`make_green_silence_data` generates the DGP: characteristics $X$ with a Toeplitz covariance, Gaussian kernel features $k(X_{ij}, P_{j\ell})$ on the decile grid $P_{j\ell} = \Phi^{-1}((\ell - 0.5)/L)$ of each characteristic, a group-sparse coefficient vector $b_0$ (so that $g_0(X) = \mathbf{k} b_0$), and a selection equation whose active set is disjoint from the outcome active set — the high-dimensional counterpart of an exclusion restriction. The intercept of the selection equation is calibrated to a target disclosure rate.

The returned `DoubleMLData` object holds the disclosure indicator in `d_cols`, the raw characteristics **and** their kernel features in `x_cols`, and the excluded variables $U$ in `z_cols`.

In [ ]:
n_obs = 4000
sim = make_green_silence_data(
    n_obs=n_obs,
    dim_x=12,
    dim_z=8,
    n_deciles=6,
    theta=-0.5,
    selection_rate=0.3,
    return_type="dict",
)
dml_data = sim["dml_data"]
print(dml_data)

In [ ]:
print(f"disclosure rate:            {sim['d'].mean():.3f}")
print(f"observed outcomes:          {int(sim['d'].sum())} of {n_obs}")
print(f"kernel features (J x L):    {sim['k'].shape[1]}")
print(f"active outcome groups:      {sim['active_outcome']}")
print(f"active selection variables: X {sim['active_selection_x']}, U {sim['active_selection_z']}")

Note that the active sets are (nearly) disjoint: the characteristics that drive *disclosure* are not the ones that drive *emissions*. This is what makes $\theta_0$ identified without hand-picking a single instrument.

## Estimation

As for every `DoubleML` class, the learners are initialized first. The DGP is sparse in the kernel features, so $\ell_1$-penalized learners are a natural choice.

In [ ]:
ml_pi = LogisticRegression(penalty="l1", C=0.1, solver="liblinear", max_iter=5000)
ml_g = LassoCV(max_iter=20000)
ml_m = LassoCV(max_iter=20000)

dml_sscf = DoubleMLSSCF(dml_data, ml_pi, ml_g, ml_m, n_folds=5, n_rep=5)
dml_sscf.fit()
print(dml_sscf)

The true value is $\theta_0 = -0.5$. A negative $\theta_0$ means that unobservables which *increase* disclosure are associated with *lower* emissions — the "green silence" pattern: firms that are clean for reasons the econometrician does not observe are the ones that choose to speak.

Because $\hat h$ is a *generated* regressor that is at the same time the *target* of $m$, the training folds are split in half by default (`nested_cross_fitting=True`): the first half estimates the selection equation, the second half estimates $g$ and $m$. This avoids own-observation bias in $\hat m$.

In [ ]:
print(dml_sscf.confint(level=0.95))

## Testing for the presence of selection bias

The class provides the doubly robust orthogonal score test of $H_0: \theta_0 = 0$, which evaluates the score at $\theta = 0$ using the cross-fitted nuisance estimates:

$$
S_n = \frac{\big(\sum_i D_i \psi(W_i; 0, \hat\eta)\big)^2}{\sum_i D_i \big(\psi(W_i; 0, \hat\eta) - \bar\psi_n\big)^2} \xrightarrow{d} \chi^2_1 .


In [ ]:
test = dml_sscf.selection_bias_test()
for key, value in test.items():
    print(f"{key:>16}: {value}")

## Quantifying the selection bias

`bias_quantification()` returns the estimated bias term $\hat\theta \hat h_i$, the corrected outcomes on the selected sample, and summary statistics. The inverse Mills ratio is much larger for the non-disclosers, so the bias implied for *them* — the units whose outcomes have to be imputed — is considerably larger than the bias among disclosers.

In [ ]:
bias = dml_sscf.bias_quantification()
summary = pd.DataFrame(
    {
        "estimated": [
            bias["summary"]["mean_imr_selected"],
            bias["summary"]["mean_imr_non_selected"],
            bias["summary"]["mean_bias_selected"],
            bias["summary"]["mean_bias_non_selected"],
        ],
        "true": [
            sim["imr"][sim["d"] == 1].mean(),
            sim["imr"][sim["d"] == 0].mean(),
            sim["theta"] * sim["imr"][sim["d"] == 1].mean(),
            sim["theta"] * sim["imr"][sim["d"] == 0].mean(),
        ],
    },
    index=["mean IMR, disclosers", "mean IMR, non-disclosers", "bias, disclosers", "bias, non-disclosers"],
)
summary.round(4)

## Comparison with the naive two-step estimator

Without the orthogonalization step — that is, regressing the outcome residual on $\hat h$ instead of on $\hat h - \hat m(X)$, and without cross-fitting — the regularization bias of the high-dimensional first stages contaminates the estimate of $\theta_0$. The simulation below shows that the naive estimator is severely attenuated and, unlike the orthogonal estimator, does not improve with the sample size.

In [ ]:
from scipy.stats import norm
from sklearn.base import clone


def naive_two_step(sim):
    """Plug-in Heckman correction without partialling out E[h | X] and without cross-fitting."""
    x = np.column_stack((sim["x"], sim["k"]))
    z_sel = np.column_stack((x, sim["u"]))
    y, d = sim["y"], sim["d"]

    pi_hat = np.clip(clone(ml_pi).fit(z_sel, d).predict_proba(z_sel)[:, 1], 1e-2, 1 - 1e-2)
    h_hat = norm.pdf(norm.ppf(pi_hat)) / pi_hat

    sel = d == 1
    g_hat = clone(ml_g).fit(x[sel], y[sel]).predict(x)
    return np.sum((y[sel] - g_hat[sel]) * h_hat[sel]) / np.sum(h_hat[sel] ** 2)


print(f"orthogonal DML: {dml_sscf.coef[0]:+.4f}")
print(f"naive two-step: {naive_two_step(sim):+.4f}")
print(f"truth:          {sim['theta']:+.4f}")

### Sampling distributions

A small Monte Carlo study (this takes a while) illustrates the difference. The orthogonal estimator is centered near the truth and its bias shrinks with the sample size; the naive estimator is inconsistent.

In [ ]:
n_rep_mc = 100
estimates = {"orthogonal DML": [], "naive two-step": []}

np.random.seed(42)
for i_rep in range(n_rep_mc):
    sim_rep = make_green_silence_data(
        n_obs=2000, dim_x=12, dim_z=8, n_deciles=6, theta=-0.5, return_type="dict"
    )
    obj = DoubleMLSSCF(sim_rep["dml_data"], ml_pi, ml_g, ml_m, n_folds=5)
    obj.fit()
    estimates["orthogonal DML"].append(obj.coef[0])
    estimates["naive two-step"].append(naive_two_step(sim_rep))

df_mc = pd.DataFrame(estimates)
df_mc.describe().round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for column, style in [("orthogonal DML", "-"), ("naive two-step", "--")]:
    values = df_mc[column].values
    bw = 1.06 * values.std(ddof=1) * len(values) ** (-1 / 5)
    grid = np.linspace(df_mc.values.min() - 0.1, df_mc.values.max() + 0.1, 300)
    dens = np.mean(np.exp(-0.5 * ((grid[:, None] - values[None, :]) / bw) ** 2), axis=1) / (bw * np.sqrt(2 * np.pi))
    ax.plot(grid, dens, style, label=column)
ax.axvline(-0.5, color="black", linewidth=1, label=r"$\theta_0$")
ax.set_xlabel(r"$\hat\theta$")
ax.set_ylabel("density")
ax.legend(frameon=False)
plt.show()

## Post variable selection

Once $\theta_0$ is consistently estimated, the outcome equation can be corrected and variable selection carried out on the corrected response $Y_i - \hat\theta \hat h_i$. The `variable_selection` helper implements the adaptive kernel group lasso: it minimizes

$$
\tfrac{1}{2}\big\|Y - \hat\theta \hat h - \mathbf{k} b\big\|^2 + \lambda \sum_{j} w_j \|b_j\|_{\mathsf{K}_j},
\qquad
w_j = \left(\frac{1 + |\hat\beta_j|}{\|\hat b_j^{\text{init}}\|_2}\right)^{\gamma},
$$

where the group norm is induced by the kernel matrix $\mathsf{K}_j$ of the sorted portfolios. The adaptive weights implement a data-driven, "soft" exclusion restriction: characteristics with a strong signal in the *selection* equation are penalized more heavily in the *outcome* equation, unless they also carry a strong outcome signal.

Setting `theta_hat=0` gives the uncorrected benchmark, so the corrected (M1) and uncorrected (M2) specifications can be compared directly.

In [ ]:
from doubleml_sscf import selection_metrics, variable_selection

sel = sim["d"] == 1
beta_hat_x = np.abs(
    clone(ml_pi)
    .fit(np.column_stack((sim["x"], sim["k"], sim["u"])), sim["d"])
    .coef_.ravel()[: sim["x"].shape[1]]
)

m1 = variable_selection(
    sim["y"][sel], sim["k"][sel], sim["groups"], sim["K_mats"], beta_hat_x,
    theta_hat=dml_sscf.coef[0], imr=dml_sscf.imr[sel],
)
m2 = variable_selection(
    sim["y"][sel], sim["k"][sel], sim["groups"], sim["K_mats"], beta_hat_x, theta_hat=0.0,
)

n_groups = len(sim["K_mats"])
print("true active groups:", sim["active_outcome"])
print("M1 (corrected)   :", m1["active_groups"])
print("M2 (uncorrected) :", m2["active_groups"])

Selection accuracy is often similar across the two specifications; the correction matters most for what the selected model is *used for* — imputing the outcomes of the non-disclosing units.

In [ ]:
k_out = sim["k"][~sel]
truth_out = k_out @ sim["b"]

rows = []
for label, res in [("M1 (corrected)", m1), ("M2 (uncorrected)", m2)]:
    pred = res["post"]["intercept"] + k_out @ res["post"]["coef"]
    met = selection_metrics(res["active_groups"], sim["active_outcome"], n_groups)
    rows.append(
        {
            "specification": label,
            "TPR": met["tpr"],
            "FPR": met["fpr"],
            "imputation bias": np.mean(pred - truth_out),
            "imputation RMSE": np.sqrt(np.mean((pred - truth_out) ** 2)),
        }
    )
pd.DataFrame(rows).round(4)

## References

Chen, C. Y.-H., Lioui, A., and Scaillet, O. Green Silence: Double machine learning carbon emissions under sample selection bias.

Chernozhukov, V., Chetverikov, D., Demirer, M., Duflo, E., Hansen, C., Newey, W., and Robins, J. (2018), Double/debiased machine learning for treatment and structural parameters, *The Econometrics Journal*, 21(1), C1-C68.

Heckman, J. J. (1979), Sample selection bias as a specification error, *Econometrica*, 47(1), 153-161.

Yuan, M. and Lin, Y. (2006), Model selection and estimation in regression with grouped variables, *Journal of the Royal Statistical Society Series B*, 68(1), 49-67.